# Détection d'attaques MitM dans un réseau de drones (IoD)
**Projet étudiant — Machine Learning centralisé + Federated Learning**

Ce notebook est organisé en 3 parties :
1. **Preprocessing** — nettoyage et préparation du dataset
2. **ML Centralisé** — Random Forest et XGBoost entraînés sur tout le dataset
3. **Federated Learning** — simulation FL avec Flower, un client par nœud réseau

---
## Partie 1 — Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Chargement ---
df = pd.read_excel('dataset_mitm_1000_rows.xlsx')
print(f'Shape : {df.shape}')
print(f"\nDistribution des labels :")
print(df['label'].value_counts())
print(f"  → {df['label'].mean()*100:.1f}% de fenêtres attaquées (label=1)")
df.head()

In [ ]:
# --- Nettoyage ---

# pkt_rate contient des dates Excel mal parsées -> on force en numérique
df['pkt_rate'] = pd.to_numeric(df['pkt_rate'], errors='coerce')

# Colonnes inutiles pour le modèle (identifiants, pas des features)
COLS_TO_DROP = ['window_start', 'src_ip', 'node']
df_clean = df.drop(columns=COLS_TO_DROP)

# Imputation des NaN par la médiane (simple et robuste)
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

print(f'Features utilisées : {[c for c in df_clean.columns if c != "label"]}')
print(f'NaN restants : {df_clean.isna().sum().sum()}')
df_clean.describe()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Séparation features / label ---
FEATURES = [c for c in df_clean.columns if c != 'label']
X = df_clean[FEATURES].values
y = df_clean['label'].values

# --- Train / Test split stratifié (garde la proportion des classes) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # important sur dataset déséquilibré
)

# --- Normalisation (utile surtout pour le MLP du FL) ---
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train.shape[0]} fenêtres  |  Test : {X_test.shape[0]} fenêtres')
print(f'Train label=0 : {(y_train==0).sum()}  |  label=1 : {(y_train==1).sum()}')

---
## Partie 2 — Machine Learning Centralisé

On teste deux modèles bien adaptés aux features tabulaires agrégées :
- **Random Forest** : simple, interprétable, supporte `class_weight='balanced'` pour le déséquilibre
- **XGBoost** : boosting, souvent meilleur sur données déséquilibrées

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)

def evaluer_modele(nom, modele, X_tr, y_tr, X_te, y_te):
    """Entraîne le modèle et affiche toutes les métriques."""
    modele.fit(X_tr, y_tr)
    y_pred  = modele.predict(X_te)
    y_proba = modele.predict_proba(X_te)[:, 1]

    print(f"\n{'='*50}")
    print(f"  {nom}")
    print(f"{'='*50}")
    print(classification_report(y_te, y_pred, target_names=['Normal (0)', 'MitM (1)']))
    print(f"ROC-AUC : {roc_auc_score(y_te, y_proba):.4f}")

    # Matrice de confusion
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(confusion_matrix(y_te, y_pred),
                           display_labels=['Normal', 'MitM']).plot(ax=ax, colorbar=False)
    ax.set_title(nom)
    plt.tight_layout()
    plt.show()

    return modele

In [ ]:
# --- Random Forest ---
# class_weight='balanced' : pondère automatiquement les classes minoritaires
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
rf = evaluer_modele('Random Forest', rf, X_train, y_train, X_test, y_test)

In [ ]:
# --- XGBoost ---
# scale_pos_weight : compense le déséquilibre (ratio classe majoritaire / minoritaire)
ratio_desequilibre = (y_train == 1).sum() / (y_train == 0).sum()
print(f'scale_pos_weight utilisé : {ratio_desequilibre:.2f}')

xgb = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=ratio_desequilibre,
    eval_metric='logloss',
    random_state=42
)
xgb = evaluer_modele('XGBoost', xgb, X_train, y_train, X_test, y_test)

In [ ]:
# --- Feature Importances (Random Forest) ---
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Importance des features — Random Forest')
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

print("\nTop 5 features les plus discriminantes :")
print(importances.sort_values(ascending=False).head())

---
## Partie 3 — Federated Learning avec Flower

### Principe
Au lieu d'entraîner sur tout le dataset centralisé, on simule le cas réel :
chaque nœud du réseau (drone ou station sol) entraîne **localement** son modèle
sur **ses propres données** et n'envoie que les **poids** au serveur central,
qui les agrège (FedAvg). Les données brutes ne quittent jamais le nœud.

```
     [UAV-1]   [UAV-2]   [UAV-3]   [ZSP]
        ↓         ↓         ↓        ↓
    train local sur données locales
        ↓         ↓         ↓        ↓
        └─────────→  SERVEUR FL  ←───┘
                    FedAvg(poids)
                    → modèle global
```

**On utilise Flower en mode simulation** (`flwr.simulation`) qui fait tourner
tout ça dans le même processus Python — pas besoin de vrais sockets réseau.

In [ ]:
import flwr as fl
from flwr.simulation import run_simulation
from flwr.client import NumPyClient
from flwr.server.strategy import FedAvg
from flwr.common import Context

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print(f'Flower version : {fl.__version__}')
print(f'PyTorch version : {torch.__version__}')

In [ ]:
# --- Partition du dataset par nœud réseau ---
# Chaque nœud (drone/station) devient un "client" FL avec ses propres données

NŒUDS = df['node'].unique().tolist()
print(f'Nœuds trouvés ({len(NŒUDS)}) : {NŒUDS}')

# On reconstruit X_scaled avec les index originaux pour pouvoir filtrer par nœud
df_model = df_clean.copy()
df_model['node'] = df['node'].values
X_all_sc = scaler.fit_transform(df_clean[FEATURES].values)  # re-fit sur tout

# Dictionnaire : node_id -> (X, y) normalisés
partitions = {}
for i, nœud in enumerate(NŒUDS):
    mask = (df['node'] == nœud).values
    partitions[i] = (
        X_all_sc[mask].astype(np.float32),
        y[mask].astype(np.float32)
    )
    print(f'  Client {i} ({nœud[:30]}) : {mask.sum()} fenêtres, '
          f"{(y[mask]==1).sum()} MitM / {(y[mask]==0).sum()} Normal")

NUM_CLIENTS = len(NŒUDS)
N_FEATURES  = X_all_sc.shape[1]

In [ ]:
# --- Définition du MLP (modèle utilisé dans le FL) ---
# Simple : 2 couches cachées, suffisant pour des features tabulaires

class MLP(nn.Module):
    def __init__(self, n_input):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()      # sortie en probabilité [0,1]
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# Vérification rapide
modele_test = MLP(N_FEATURES)
x_test = torch.rand(8, N_FEATURES)
print(f'MLP OK — sortie shape : {modele_test(x_test).shape}')  # attendu : torch.Size([8])

In [ ]:
# --- Fonctions utilitaires : extraire / injecter les poids du MLP ---
# Flower échange les poids sous forme de listes numpy ("NDArrays")

def get_weights(model):
    """Extrait les poids du modèle PyTorch -> liste de numpy arrays."""
    return [p.detach().cpu().numpy() for p in model.parameters()]

def set_weights(model, weights):
    """Injecte une liste de numpy arrays dans le modèle PyTorch."""
    for param, w in zip(model.parameters(), weights):
        param.data = torch.tensor(w)

def entrainer_local(model, X, y, epochs=5, lr=0.01):
    """Entraîne le modèle sur les données locales du client."""
    dataset   = TensorDataset(torch.tensor(X), torch.tensor(y))
    loader    = DataLoader(dataset, batch_size=32, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # pos_weight : compense le déséquilibre local (si toutes les fenêtres sont MitM)
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    pos_w = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    model.train()
    total_loss = 0
    for _ in range(epochs):
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = loss_fn(pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    return total_loss

def evaluer_local(model, X, y):
    """Évalue le modèle sur des données (pas de gradient)."""
    model.eval()
    with torch.no_grad():
        pred    = model(torch.tensor(X))
        labels  = torch.tensor(y)
        loss    = nn.BCELoss()(pred, labels).item()
        correct = ((pred > 0.5).float() == labels).sum().item()
    return loss, correct / len(y)

In [ ]:
# --- Définition du Client Flower ---
# Chaque client représente un nœud (drone/ZSP) avec ses données locales

class ClientMitM(NumPyClient):
    def __init__(self, client_id, X_local, y_local, n_features):
        self.client_id = client_id
        self.X = X_local
        self.y = y_local
        self.model = MLP(n_features)   # modèle local, initialisé aléatoirement

    def get_parameters(self, config):
        """Le serveur demande les poids actuels du client."""
        return get_weights(self.model)

    def fit(self, parameters, config):
        """Reçoit les poids globaux, entraîne localement, renvoie les nouveaux poids."""
        set_weights(self.model, parameters)          # 1. Charger le modèle global
        entrainer_local(self.model, self.X, self.y)  # 2. Entraîner sur données locales
        return get_weights(self.model), len(self.X), {}  # 3. Renvoyer poids + nb exemples

    def evaluate(self, parameters, config):
        """Évalue le modèle global (reçu du serveur) sur les données locales."""
        set_weights(self.model, parameters)
        loss, acc = evaluer_local(self.model, self.X, self.y)
        return loss, len(self.X), {'accuracy': acc}


# Factory function : Flower appelle cette fonction pour créer chaque client
def client_factory(context: Context) -> NumPyClient:
    client_id = int(context.node_config['partition-id'])
    X_local, y_local = partitions[client_id]
    return ClientMitM(client_id, X_local, y_local, N_FEATURES)

print('Client FL défini.')

In [ ]:
# --- Lancement de la simulation FL ---
# FedAvg : agrège les poids de tous les clients en faisant une moyenne pondérée
#          par le nombre d'exemples de chaque client

strategie = FedAvg(
    fraction_fit=1.0,            # 100% des clients participent à chaque round
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
)

historique = run_simulation(
    server_app=fl.server.ServerApp(
        config=fl.server.ServerConfig(num_rounds=10),  # 10 rounds de communication
        strategy=strategie,
    ),
    client_app=fl.client.ClientApp(client_fn=client_factory),
    num_supernodes=NUM_CLIENTS,
    backend_config={"client_resources": {"num_cpus": 1}},
)

print('\nSimulation FL terminée.')

In [ ]:
# --- Courbe de convergence FL ---

# Flower stocke les métriques d'évaluation distribuée dans losses_distributed
losses = [v for _, v in historique.losses_distributed]

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(losses)+1), losses, marker='o', color='coral')
plt.xlabel('Round FL')
plt.ylabel('Loss moyenne (BCE)')
plt.title('Convergence du modèle global — Federated Learning (FedAvg)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- Évaluation finale du modèle FL sur le test set centralisé ---
# Pour comparer équitablement avec RF et XGBoost

from sklearn.metrics import classification_report, roc_auc_score

# Récupérer les derniers poids agrégés (dernier round)
# On recrée un MLP et on injecte les poids du round final
modele_fl_final = MLP(N_FEATURES)

# Les poids du dernier round sont dans parameters_aggregated
derniers_poids = historique.parameters_aggregated

if derniers_poids is not None:
    from flwr.common import parameters_to_ndarrays
    set_weights(modele_fl_final, parameters_to_ndarrays(derniers_poids))

    modele_fl_final.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test_sc.astype(np.float32))
        proba_fl = modele_fl_final(X_test_tensor).numpy()
        pred_fl  = (proba_fl > 0.5).astype(int)

    print('\n' + '='*50)
    print('  MLP Federated Learning (FedAvg, 10 rounds)')
    print('='*50)
    print(classification_report(y_test, pred_fl, target_names=['Normal (0)', 'MitM (1)']))
    print(f'ROC-AUC : {roc_auc_score(y_test, proba_fl):.4f}')
else:
    print("Poids FL non disponibles — relancer la simulation.")

In [ ]:
# --- Tableau récapitulatif : Centralisé vs FL ---
from sklearn.metrics import f1_score

resultats = {}

# RF
pred_rf = rf.predict(X_test)
resultats['Random Forest'] = {
    'F1 macro'  : f1_score(y_test, pred_rf, average='macro'),
    'F1 Normal' : f1_score(y_test, pred_rf, pos_label=0),
    'F1 MitM'   : f1_score(y_test, pred_rf, pos_label=1),
    'ROC-AUC'   : roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]),
}

# XGB
pred_xgb = xgb.predict(X_test)
resultats['XGBoost'] = {
    'F1 macro'  : f1_score(y_test, pred_xgb, average='macro'),
    'F1 Normal' : f1_score(y_test, pred_xgb, pos_label=0),
    'F1 MitM'   : f1_score(y_test, pred_xgb, pos_label=1),
    'ROC-AUC'   : roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]),
}

# FL (si disponible)
if derniers_poids is not None:
    resultats['MLP FL (FedAvg)'] = {
        'F1 macro'  : f1_score(y_test, pred_fl, average='macro'),
        'F1 Normal' : f1_score(y_test, pred_fl, pos_label=0),
        'F1 MitM'   : f1_score(y_test, pred_fl, pos_label=1),
        'ROC-AUC'   : roc_auc_score(y_test, proba_fl),
    }

df_resultats = pd.DataFrame(resultats).T.round(4)
print('\n=== COMPARAISON FINALE ===')
print(df_resultats.to_string())
df_resultats